In [ ]:
import os
import cv2
import mediapipe as mp
from rembg import remove, new_session
from PIL import Image
from tqdm import tqdm
import onnxruntime as ort
import numpy as np

# === Paths ===
input_dir = "fake_dataset/09000"
output_dir = "fake_dataset/cropped_faces"
os.makedirs(output_dir, exist_ok=True)
batch_size = 16

# === MediaPipe Face Detection (always CPU) ===
mp_face_detection = mp.solutions.face_detection

# === Auto GPU session selection for rembg ===
ort.set_default_logger_severity(3)  # Suppress ONNX spam
available_providers = ort.get_available_providers()

use_gpu = 'CUDAExecutionProvider' in available_providers
print(f"rembg will use: {'GPU' if use_gpu else 'CPU'}")

session = new_session("u2net", providers=["CUDAExecutionProvider" if use_gpu else "CPUExecutionProvider"])

# === Helper: Crop face using MediaPipe
def crop_face(image):
    height, width, _ = image.shape
    results = face_detector.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    if results.detections:
        d = results.detections[0]
        bbox = d.location_data.relative_bounding_box
        x, y, w, h = bbox.xmin * width, bbox.ymin * height, bbox.width * width, bbox.height * height
        pad = 0.2
        x1 = max(int(x - w * pad), 0)
        y1 = max(int(y - h * pad), 0)
        x2 = min(int(x + w * (1 + pad)), width)
        y2 = min(int(y + h * (1 + pad)), height)
        return image[y1:y2, x1:x2]
    return None

# === Main Loop ===
file_list = os.listdir(input_dir)
images_batch = []
names_batch = []

with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as face_detector:
    for fname in tqdm(file_list, desc="🔄 Batching for rembg", ncols=80):
        try:
            path = os.path.join(input_dir, fname)
            img = cv2.imread(path)
            if img is None:
                continue

            face_crop = crop_face(img)
            if face_crop is None:
                tqdm.write(f"❌ No face detected in {fname}")
                continue

            pil_image = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
            images_batch.append(pil_image)
            names_batch.append(fname)

            # === Process batch when full ===
            if len(images_batch) == batch_size:
                for img, name in zip(images_batch, names_batch):
                    out_img = remove(img, session=session)
                    save_path = os.path.join(output_dir, os.path.splitext(name)[0] + ".png")
                    out_img.save(save_path)
                images_batch.clear()
                names_batch.clear()

        except Exception as e:
            tqdm.write(f"⚠️ Error on {fname}: {e}")

    # === Final leftover batch
    if images_batch:
        for img, name in zip(images_batch, names_batch):
            out_img = remove(img, session=session)
            save_path = os.path.join(output_dir, os.path.splitext(name)[0] + ".png")
            out_img.save(save_path)



In [1]:
import os

output_dir = "fake_dataset/cropped_faces"
total_images = len([
    f for f in os.listdir(output_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print(f"Total cropped images: {total_images}")

Total cropped images: 9596


In [2]:
import os
import shutil
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from PIL import Image
import numpy as np
import torch
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# === Paths ===
input_dir = "fake_dataset/cropped_faces"
output_dir = "fake_dataset/images_clustered"
os.makedirs(output_dir, exist_ok=True)

# === Load CLIP model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# === Extract features ===
features = []
filenames = []

for fname in tqdm(os.listdir(input_dir)):
    fpath = os.path.join(input_dir, fname)
    try:
        image = Image.open(fpath).convert("RGB").resize((224, 224))
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = model.get_image_features(**inputs)
        features.append(emb.cpu().numpy().flatten())
        filenames.append(fname)
    except Exception as e:
        print(f"Skipped {fname}: {e}")

features = np.vstack(features)
scaled = StandardScaler().fit_transform(features)

# === Cluster (KMeans) ===
kmeans = KMeans(n_clusters=4, random_state=42)
labels = kmeans.fit_predict(scaled)

# === Save clustered images ===
for cluster_id in range(4):
    os.makedirs(os.path.join(output_dir, f"Cluster_{cluster_id}"), exist_ok=True)

for fname, label in zip(filenames, labels):
    src = os.path.join(input_dir, fname)
    dst = os.path.join(output_dir, f"Cluster_{label}", fname)
    shutil.copyfile(src, dst)

print("Clustering completed. Check the folders under:", output_dir)

total_clustered = sum(
    len(files) for _, _, files in os.walk(output_dir)
)
print(f"Total clustered images saved: {total_clustered}")

D:\001_MLProjects\color_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\001_MLProjects\color_analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
D:\001_MLProjects\color_analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 9596/9596 [07:20<00:00, 21.76it/s]


Clustering completed. Check the folders under: fake_dataset/images_clustered
Total clustered images saved: 9596


In [3]:
print("\n📊 Images per cluster:")
for cluster_id in range(4):
    folder = os.path.join(output_dir, f"Cluster_{cluster_id}")
    count = len([
        f for f in os.listdir(folder)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])
    print(f"  Cluster_{cluster_id}: {count} images")


📊 Images per cluster:
  Cluster_0: 3062 images
  Cluster_1: 1499 images
  Cluster_2: 2206 images
  Cluster_3: 2829 images


In [6]:
import os
import cv2
import numpy as np
from scipy.spatial.distance import euclidean
import itertools
from tqdm import tqdm
import shutil

# === Feature Extraction ===
def extract_hsv_features(folder):
    hsv_features = []
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        img = cv2.imread(fpath)
        if img is None:
            continue
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mean_hsv = np.mean(hsv.reshape(-1, 3), axis=0)
        hist = cv2.calcHist([hsv], [0], None, [8], [0, 180]).flatten()
        hist /= hist.sum()
        feature = np.concatenate([mean_hsv, hist])
        hsv_features.append(feature)
    if len(hsv_features) == 0:
        return np.zeros(11)
    return np.mean(hsv_features, axis=0)

# === Paths ===
cluster_root = "fake_dataset/images_clustered"
season_root = "fake_dataset/main_season_ai_data"

clusters = sorted([d for d in os.listdir(cluster_root) if d.startswith("Cluster")])
seasons = ["autumn", "spring", "summer", "winter"]

# === Ensure all season folders exist ===
for s in seasons:
    path = os.path.join(season_root, s)
    if not os.path.exists(path):
        os.makedirs(path)
        print(f"📁 Created missing season folder: {path}")

# === Extract features ===
print("🔍 Extracting features...")
cluster_features = {
    c: extract_hsv_features(os.path.join(cluster_root, c)) for c in tqdm(clusters)
}
season_features = {
    s: extract_hsv_features(os.path.join(season_root, s)) for s in seasons
}

# === Find best unique mapping ===
best_mapping = {}
best_total_dist = float("inf")

for perm in itertools.permutations(seasons):
    total_dist = 0
    temp_map = {}
    for c, s in zip(clusters, perm):
        dist = euclidean(cluster_features[c], season_features[s])
        total_dist += dist
        temp_map[c] = s
    if total_dist < best_total_dist:
        best_total_dist = total_dist
        best_mapping = temp_map

# === Print and Copy Images ===
print("\n✅ Best Cluster-to-Season Mapping (unique):")
for cluster, season in best_mapping.items():
    print(f"  {cluster} → {season}")
    src_dir = os.path.join(cluster_root, cluster)
    dst_dir = os.path.join(season_root, season)

    for fname in os.listdir(src_dir):
        src_path = os.path.join(src_dir, fname)
        dst_path = os.path.join(dst_dir, fname)
        shutil.copyfile(src_path, dst_path)

print("\n📦 Images copied to their corresponding season folders.")

🔍 Extracting features...


100%|██████████| 4/4 [03:33<00:00, 53.39s/it]



✅ Best Cluster-to-Season Mapping (unique):
  Cluster_0 → autumn
  Cluster_1 → spring
  Cluster_2 → summer
  Cluster_3 → winter

📦 Images copied to their corresponding season folders.


In [10]:
import os
import shutil
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from PIL import Image
import numpy as np
import torch
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# === Settings ===
input_dir = "main_season_ai_data/autumn"
output_base = "fake_dataset/main_season_ai_data/autumn"
n_subclusters = 3

# === Load CLIP model ===
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# === Extract features ===
features = []
filenames = []

for fname in tqdm(os.listdir(input_dir)):
    fpath = os.path.join(input_dir, fname)
    try:
        image = Image.open(fpath).convert("RGB").resize((224, 224))
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            emb = model.get_image_features(**inputs)
        features.append(emb.cpu().numpy().flatten())
        filenames.append(fname)
    except Exception as e:
        print(f"Skipped {fname}: {e}")

features = np.vstack(features)
scaled = StandardScaler().fit_transform(features)

# === KMeans Sub-clustering ===
kmeans = KMeans(n_clusters=n_subclusters, random_state=42)
labels = kmeans.fit_predict(scaled)

# === Save to subfolders ===
for sub_id in range(n_subclusters):
    os.makedirs(os.path.join(output_base, f"Sub_{sub_id}"), exist_ok=True)

for fname, label in zip(filenames, labels):
    src = os.path.join(input_dir, fname)
    dst = os.path.join(output_base, f"Sub_{label}", fname)
    shutil.move(src, dst)

print("Winter sub-clustering complete.")

cuda


D:\001_MLProjects\color_analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 3062/3062 [02:28<00:00, 20.63it/s]


Winter sub-clustering complete.
